# 04 Gold — card meta metrics

The report layer. Aggregates the silver card-grain table and quality check
table into two small, dashboard-ready Delta tables:

- **`gold_card_metrics`** — one row per card: pick rate, win rate, sample size,
  enriched with `elixir_band` so the dashboard can cut by cost tier.
- **`gold_overview`** — a single KPI row for the dashboard's scalar tiles
  (totals, freshness, validity).

**Meta scope:** Trophy ladder only (`is_ladder = true`). Path of Legend and the
other modes are deliberately excluded so win/pick rates describe one metagame.

In [0]:
from pyspark.sql import functions as F, types as T

CATALOG, SCHEMA = "workspace", "clash"

# Sources
SILVER_BATTLES     = f"{CATALOG}.{SCHEMA}.silver_battles"
SILVER_DECK_CARDS  = f"{CATALOG}.{SCHEMA}.silver_deck_cards"
DQ_RESULTS         = f"{CATALOG}.{SCHEMA}.dq_results"
QUARANTINE_BATTLES = f"{CATALOG}.{SCHEMA}.quarantine_battles"

# Targets (gold)
GOLD_CARD_METRICS = f"{CATALOG}.{SCHEMA}.gold_card_metrics"
GOLD_OVERVIEW     = f"{CATALOG}.{SCHEMA}.gold_overview"

SCOPE_LABEL = "trophy_ladder"   # is_ladder = true
print("scope:", SCOPE_LABEL)

scope: trophy_ladder


## `gold_card_metrics`

`silver_deck_cards` is already at card grain: one row per card, per side, per
battle. For this layer we scope it to trophy ladder mode only, but all the
modes are kept in the database for other extended analysis:

- **pick rate** = appearances of a card / total deck instances (each battle has
  two decks, so the denominator is `2 × ladder battles`, minus any missing decks).
- **win rate** = that card's side wins / its appearances. `won` is the side's
  outcome from crown counts; equal-crown draws (~0.1%) count as non-wins.

`n_appearances` is kept as a plain column so the dashboard can filter out
tiny-sample cards instead of needing a confidence interval.

In [0]:
# Card-grain rows, scoped to trophy ladder.
ladder_cards = spark.table(SILVER_DECK_CARDS).filter("is_ladder = true")

# Pick-rate denominator: distinct deck instances (battle_id x side) in scope.
total_decks = ladder_cards.select("battle_id", "side").distinct().count()
print("deck instances in scope:", total_decks)

gold_card = (ladder_cards
    .groupBy("card_id", "card_name", "rarity", "elixir_cost", "elixir_band")
    .agg(
        F.count("*").alias("n_appearances"),
        # count(won) ignores nulls, so the win-rate denominator only counts
        # battles whose outcome was decidable (crowns present on both sides).
        F.count("won").alias("n_decided"),
        F.sum(F.col("won").cast("int")).alias("wins"),
    )
    .withColumn("pick_rate", F.col("n_appearances") / F.lit(total_decks))
    .withColumn("win_rate", F.col("wins") / F.col("n_decided"))
    .withColumn("scope", F.lit(SCOPE_LABEL))
    .withColumn("generated_at", F.current_timestamp()))

(gold_card.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_CARD_METRICS))

gc = spark.table(GOLD_CARD_METRICS)
print(gc.count(), "cards in gold_card_metrics")
display(gc.orderBy(F.desc("pick_rate")).limit(10))

deck instances in scope: 227668
121 cards in gold_card_metrics


card_id,card_name,rarity,elixir_cost,elixir_band,n_appearances,n_decided,wins,pick_rate,win_rate,scope,generated_at
28000011,The Log,legendary,2,cheap,64873,64873,33768,0.28494562257322065,0.5205247175250104,trophy_ladder,2026-06-10T07:33:20.688Z
28000001,Arrows,common,3,mid,63352,63352,31111,0.27826484178716376,0.49108157595656016,trophy_ladder,2026-06-10T07:33:20.688Z
28000000,Fireball,rare,4,mid,53688,53688,28185,0.23581706695714813,0.5249776486365668,trophy_ladder,2026-06-10T07:33:20.688Z
28000008,Zap,common,2,cheap,46847,46847,23777,0.20576892668271343,0.5075458407155208,trophy_ladder,2026-06-10T07:33:20.688Z
26000055,Mega Knight,legendary,7,heavy,46238,46238,21907,0.20309397895180703,0.47378779358968814,trophy_ladder,2026-06-10T07:33:20.688Z
26000021,Hog Rider,rare,4,mid,45531,45531,23115,0.1999885798619042,0.5076760888186072,trophy_ladder,2026-06-10T07:33:20.688Z
26000000,Knight,common,3,mid,42866,42866,21798,0.18828293831368484,0.5085149069192367,trophy_ladder,2026-06-10T07:33:20.688Z
26000011,Valkyrie,rare,4,mid,42538,42538,20074,0.1868422439692886,0.47190747096713526,trophy_ladder,2026-06-10T07:33:20.688Z
26000010,Skeletons,common,1,cheap,38236,38236,20074,0.1679463077815064,0.5250026153363323,trophy_ladder,2026-06-10T07:33:20.688Z
28000015,Barbarian Barrel,epic,2,cheap,36907,36907,20355,0.16210886027021804,0.5515213916059284,trophy_ladder,2026-06-10T07:33:20.688Z


## `gold_overview`

One KPI row for the dashboard's scalar tiles. Counts come from silver;
freshness and validity are read from the latest `dq_results` run rather than
recomputed, so the quality layer stays the single source of truth.

- `distinct_participants` — every tag seen on either side. Mostly one-off
  opponents we never crawled, so it is far larger than the ~10k seed players.
- `validity_pct` — share of battles that passed every *error* check this run
  (i.e. were not quarantined).

In [0]:
sb = spark.table(SILVER_BATTLES)
total_battles  = sb.count()
ladder_battles = sb.filter("is_ladder = true").count()

# Distinct participants: union of both sides\' tags (dominated by one-off opponents).
participants = (sb.select(F.col("player_tag").alias("tag"))
    .union(sb.select(F.col("opponent_tag").alias("tag")))
    .filter(F.col("tag").isNotNull()).distinct().count())

distinct_cards = spark.table(GOLD_CARD_METRICS).count()

# Freshness + validity from the latest dq_results run (single source of truth).
latest_run = spark.table(DQ_RESULTS).agg(F.max("run_id")).first()[0]
dq = spark.table(DQ_RESULTS).filter(F.col("run_id") == latest_run)
freshness_hours = (dq.filter("check_name = 'freshness_under_24h'")
    .agg(F.max("metric_value")).first()[0])

quarantined = spark.table(QUARANTINE_BATTLES).select("battle_id").distinct().count()
validity_pct = 100.0 * (1 - quarantined / total_battles) if total_battles else None

overview_schema = T.StructType([
    T.StructField("total_battles", T.LongType()),
    T.StructField("ladder_battles", T.LongType()),
    T.StructField("distinct_participants", T.LongType()),
    T.StructField("distinct_cards", T.LongType()),
    T.StructField("freshness_hours", T.DoubleType()),
    T.StructField("validity_pct", T.DoubleType()),
    T.StructField("scope", T.StringType()),
    T.StructField("dq_run_id", T.StringType()),
])
overview = spark.createDataFrame([(
    total_battles, ladder_battles, participants, distinct_cards,
    float(freshness_hours) if freshness_hours is not None else None,
    float(validity_pct) if validity_pct is not None else None,
    SCOPE_LABEL, latest_run,
)], overview_schema).withColumn("generated_at", F.current_timestamp())

(overview.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_OVERVIEW))
display(spark.table(GOLD_OVERVIEW))

total_battles,ladder_battles,distinct_participants,distinct_cards,freshness_hours,validity_pct,scope,dq_run_id,generated_at
603439,113834,418595,121,40.31635463777778,93.55361519557071,trophy_ladder,20260609T201452,2026-06-10T07:57:06.372Z


## Dashboard preview — the two headline charts

Top cards by win rate (with a minimum-sample floor so a rarely-played card can't
spike the chart) and top cards by pick rate.

In [0]:
gc = spark.table(GOLD_CARD_METRICS)

MIN_APPEARANCES = 500   # floor out tiny-sample cards from the win-rate chart
print(f"top cards by win rate (>= {MIN_APPEARANCES} appearances)")
display(gc.filter(F.col("n_appearances") >= MIN_APPEARANCES)
    .orderBy(F.desc("win_rate"))
    .select("card_name", "rarity", "elixir_band", "win_rate", "pick_rate", "n_appearances")
    .limit(10))

print("top cards by pick rate")
display(gc.orderBy(F.desc("pick_rate"))
    .select("card_name", "rarity", "elixir_band", "pick_rate", "win_rate", "n_appearances")
    .limit(10))

top cards by win rate (>= 500 appearances)


card_name,rarity,elixir_band,win_rate,pick_rate,n_appearances
Zappies,rare,mid,0.5588483442311749,0.04164836516330798,9482
Rascals,common,heavy,0.5587225929456625,0.04607586485584272,10490
Cannon Cart,epic,heavy,0.5566521910777734,0.033377549765447934,7599
Mortar,common,mid,0.5555208333333334,0.042166663738426126,9600
Goblin Cage,rare,mid,0.5517832367583448,0.053819596956972436,12253
Barbarian Barrel,epic,cheap,0.5515213916059284,0.16210886027021804,36907
Royal Ghost,legendary,mid,0.5492134831460674,0.06841101955479031,15575
Skeleton Dragons,common,mid,0.5439814814814815,0.024667498286979287,5616
Bomb Tower,rare,mid,0.5432595573440644,0.024013036526872464,5467
Royal Recruits,common,heavy,0.5425898137523171,0.04976105557214892,11329


top cards by pick rate


card_name,rarity,elixir_band,pick_rate,win_rate,n_appearances
The Log,legendary,cheap,0.28494562257322065,0.5205247175250104,64873
Arrows,common,mid,0.27826484178716376,0.49108157595656016,63352
Fireball,rare,mid,0.23581706695714813,0.5249776486365668,53688
Zap,common,cheap,0.20576892668271343,0.5075458407155208,46847
Mega Knight,legendary,heavy,0.20309397895180703,0.47378779358968814,46238
Hog Rider,rare,mid,0.1999885798619042,0.5076760888186072,45531
Knight,common,mid,0.18828293831368484,0.5085149069192367,42866
Valkyrie,rare,mid,0.1868422439692886,0.47190747096713526,42538
Skeletons,common,cheap,0.1679463077815064,0.5250026153363323,38236
Barbarian Barrel,epic,cheap,0.16210886027021804,0.5515213916059284,36907
